In [10]:
# DICOM 처리를 위한 백엔드 라이브러리 설치
#!pip install pydicom itk nibabel monai torch matplotlib

In [11]:
import os
import glob
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

import monai
from monai.transforms import (
    Compose, LoadImageD, EnsureChannelFirstD, ScaleIntensityRangeD, 
    ResizeD, RandRotateD, RandFlipD, ToTensorD
)
from monai.data import Dataset
from monai.networks.nets import UNet
from monai.losses import DiceLoss

# 1. .dcm 파일 경로 설정
data_dir = "./dataset-dcm/train/gather"
# X-ray 원본은 .dcm, 라벨(마스크)은 보통 .png 또는 똑같이 .dcm일 수 있습니다. 
# 여기서는 라벨이 .png라고 가정했으나, 둘 다 .dcm이어도 무방합니다.
image_paths = sorted(glob.glob(os.path.join(data_dir, "images", "*.dcm")))
mask_paths = sorted(glob.glob(os.path.join(data_dir, "masks", "*.png"))) 

data_dicts = [{"image": img, "label": msk} for img, msk in zip(image_paths, mask_paths)]

# 2. DICOM 최적화 전처리 파이프라인
train_transforms = Compose([
    # reader="ITKReader"를 명시하여 dcm 확장자를 처리합니다.
    LoadImageD(keys=["image", "label"], reader="ITKReader"),
    EnsureChannelFirstD(keys=["image", "label"]),
    
    # DICOM의 뼈(Bone) 조직이 잘 표현되도록 윈도잉(Windowing) 및 0~1 정규화 진행
    # 뼈 조직을 강조하기 위해 대략 -200 ~ 1000 범위를 지정 (데이터 특성에 맞게 조정 가능)
    ScaleIntensityRangeD(
        keys=["image"], 
        a_min=-200, a_max=1000, 
        b_min=0.0, b_max=1.0, 
        clip=True
    ),
    
    # 2D X-ray 크기 통일
    ResizeD(keys=["image", "label"], spatial_size=(512, 512), mode=("bilinear", "nearest")),
    
    # 데이터 증강
    RandRotateD(keys=["image", "label"], range_x=0.15, prob=0.5, mode=("bilinear", "nearest")),
    RandFlipD(keys=["image", "label"], spatial_axis=1, prob=0.5),
    ToTensorD(keys=["image", "label"])
])

train_ds = Dataset(data=data_dicts, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)

print(f"DICOM 데이터셋 로드 완료: 총 {len(train_ds)}개")

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

loss_function = DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# --- 학습 진행 (샘플 코드는 생략, 이전 코드로 실행하시면 됩니다) ---

# 시각화 검증을 위해 1개 배치 가져오기
model.eval()
with torch.no_grad():
    for batch_data in train_loader:
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)
        outputs = model(inputs)
        outputs = (torch.sigmoid(outputs) > 0.5).float()
        break

# DICOM 이미지는 로드 방식에 따라 [H, W]가 아닌 [W, H]로 배치될 수 있으므로
# 시각화할 때 .squeeze().T 혹은 적절히 2D 형태로 변경해 줍니다.
img_to_show = inputs[0, 0].cpu().numpy()
lbl_to_show = labels[0, 0].cpu().numpy()
pred_to_show = outputs[0, 0].cpu().numpy()

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title("DICOM Original (Windowed)")
plt.imshow(img_to_show, cmap="gray")

plt.subplot(1, 3, 2)
plt.title("Ground Truth")
plt.imshow(lbl_to_show, cmap="gray")

plt.subplot(1, 3, 3)
plt.title("Predicted Lumbar")
plt.imshow(pred_to_show, cmap="gray")
plt.show()

In [12]:
import mimetypes
import os
import SimpleITK as sitk
import numpy as np
import PIL.Image as Image

# 3D Slicer에서 저장한 NRRD 또는 NIfTI 파일 경로
slicer_mask_path = "./dataset-dcm/train/gather_mask/Segmentation-Lumbar_mask-label.nrrd"  # 또는 .nii
# 딥러닝 학습 코드가 읽을 PNG 저장 경로
output_png_path = "./dataset-dcm/train/gather_mask/img001.png"

# 의료 영상 파일 읽기
image = sitk.ReadImage(slicer_mask_path)
mask_array = sitk.GetArrayFromImage(image)

# 3D Slicer는 2D X-ray도 내부적으로는 3D(깊이 1인 배열)로 처리할 수 있으므로 2D로 압축
if len(mask_array.shape) == 3:
    mask_array = mask_array[0]  # 첫 번째 슬라이스 추출

# 인공지능 학습을 위해 0과 255(흰색)로 변환
mask_array = (mask_array > 0).astype(np.uint8) * 255

# PNG 이미지로 저장
out_img = Image.fromarray(mask_array)
out_img.save(output_png_path)
print(f"성공적으로 PNG 변환 완료: {output_png_path}")

성공적으로 PNG 변환 완료: ./dataset-dcm/train/gather_mask/img001.png
